In [55]:
'''
python3 -m venv venv
source venv/bin/activate
pip install pandas numpy scikit-learn torch transformers biopython
'''

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, max_error, mean_absolute_error
import torch
from transformers import AutoTokenizer, AutoModel
from xgboost import XGBRegressor

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Users/dajanakolarova/PycharmProjects/ML_pH_opt/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <1A0D8152-BF46-3BE0-B651-EE965C187777> /Users/dajanakolarova/PycharmProjects/ML_pH_opt/.venv/lib/python3.13/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Crypt

In [31]:
df = pd.read_csv('pHopt_data.csv')
print(df.head())

   Unnamed: 0   Accession                       Organism  EC Number  \
0           1      P21673                   Homo sapiens   2.3.1.57   
1           2      I3WU34     Exiguobacterium acetylicum  3.2.1.142   
2           3  A0A0B5JT51            Exiguobacterium sp.   3.2.1.41   
3           4  A0A1V0FWX7  Geobacillus thermocatenulatus   3.2.1.41   
4           5      P52902                  Pisum sativum  1.2.1.104   

   Sample Weight  pHopt     Split Test <20% to Train  \
0          0.366   7.47  Training                NaN   
1          0.366   6.00  Training                NaN   
2          0.366   8.50  Training                NaN   
3          0.366   6.50  Training                NaN   
4          0.366   7.60  Training                NaN   

                                            Sequence  
0  MAKFVIRPATAADCSDILRLIKELAKYEYMEEQVILTEKDLLEDGF...  
1  MKRIRSVCIVALTFALIFSGLSLSGQALEKGKSTLVIIHYKEDPNA...  
2  MNRLKSVCAVVLTFALIFSLFPVNSLALEKGKSTLVIIHYKEDKTS...  
3  MLHISRTFAAYLD

In [45]:
# model ESM-2 
model_name = "facebook/esm2_t30_150M_UR50D" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
esm_model = AutoModel.from_pretrained(model_name)

def get_esm_embedding(sequence):
    """
    převed protein seq na vektor pomocí ESM-2
    """
    inputs = tokenizer(sequence, return_tensors="pt", add_special_tokens=True) # tokenizace
    
    with torch.no_grad():           # vypnu gradienty model netrénujeme
        outputs = esm_model(**inputs)
        
    last_hidden_states = outputs.last_hidden_state #skrytý stavy - vektory pro aa

    sequence_embedding = last_hidden_states.mean(dim=1).squeeze().numpy()  # mean Pooling - 1D vektor    
    
    return sequence_embedding

Loading weights: 100%|██████████| 486/486 [00:00<00:00, 17746.94it/s]
[transformers] EsmModel LOAD REPORT from: facebook/esm2_t30_150M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [46]:
# seq -> vektor
X_embeddings = df['Sequence'].apply(get_esm_embedding)
# 2D matice
X_features_esm = pd.DataFrame(X_embeddings.tolist())
features_esm = list(X_features_esm.columns) 

df_model_esm = pd.concat([X_features_esm, df[['pHopt', 'Split']]], axis=1)

In [47]:
'''
def calculate_sample_weights(y_values):
     #'bin-inverse' ze studie
    y = np.array(y_values)
    
    acidic_mask = y <= 5
    neutral_mask = (y > 5) & (y < 9)
    alkaline_mask = y >= 9
    
    w_acidic = 1.0 / np.sum(acidic_mask) if np.sum(acidic_mask) > 0 else 0
    w_neutral = 1.0 / np.sum(neutral_mask) if np.sum(neutral_mask) > 0 else 0
    w_alkaline = 1.0 / np.sum(alkaline_mask) if np.sum(alkaline_mask) > 0 else 0
    
    weights = np.zeros(len(y))
    weights[acidic_mask] = w_acidic
    weights[neutral_mask] = w_neutral
    weights[alkaline_mask] = w_alkaline
    
    weights = weights / np.mean(weights)
    return weights
'''

In [48]:
train_data_esm = df_model_esm[df_model_esm['Split'] == 'Training']
test_data_esm = df_model_esm[df_model_esm['Split'] == 'Testing']

X_test_esm = test_data_esm[features_esm]
y_test_esm = test_data_esm['pHopt']

X_train_esm = train_data_esm[features_esm]
y_train_esm = train_data_esm['pHopt']

#train_weights = calculate_sample_weights(y_train_esm)

In [51]:
rf_model_esm = RandomForestRegressor()

param_grid_rf_esm = {
    'n_estimators': [500], 
    'max_features': ['sqrt'], 
    'max_depth': [20], 
    'min_samples_split': [1, 2], 
    'min_samples_leaf': [1, 2]
}

grid_search_rf_esm = GridSearchCV(
    estimator=rf_model_esm, 
    param_grid=param_grid_rf_esm, 
    cv=5, 
    scoring='neg_mean_squared_error', 
    n_jobs=-1
)

grid_search_rf_esm.fit(X_train_esm, y_train_esm, sample_weight=train_weights)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/queues.py:120: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for 

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestRegressor()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'max_depth': [20], 'max_features': ['sqrt'], 'min_samples_leaf': [1, 2], 'min_samples_split': [1, 2], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter ca

In [52]:
best_rf_esm = grid_search_rf_esm.best_estimator_
y_pred_esm = best_rf_esm.predict(X_test_esm)

mse_esm = mean_squared_error(y_test_esm, y_pred_esm)
rmse = np.sqrt(mse_esm)
mae = mean_absolute_error(y_test_esm, y_pred_esm)
max_err = max_error(y_test_esm, y_pred_esm)

r2_esm = r2_score(y_test_esm, y_pred_esm)
r2_train = r2_score(y_train_esm, best_rf_esm.predict(X_train_esm))

print(f"Best params RF : {grid_search_rf_esm.best_params_}")
print(f"MSE: {mse_esm:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"MAE: {mae:.3f}")
print(f"Max error: {max_err:.3f}")
print(f"R^2: {r2_esm:.3f}")
print(f"R^2 trénovacích dat: {r2_train:.3f}")

Best params RF : {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 500}
MSE: 0.764
RMSE: 0.874
MAE: 0.623
Max error: 4.793
R^2: 0.427
R^2 trénovacích dat: 0.897


In [53]:
sekvence_lig = "MNTQRKYGRTWHYPFSPGTTSDDRINTDYWQDLKAITQLVHTEKLDGENNCLNRYGVFARSHATPTQSAWTYKIRQRWQLLKNDLGDLELFGENLYAVHSIEYRALEQDFYLFAVRCQDMWLSWEEVQFYAALFDFPCVPEISGPQPGNDEKSWQRDFLALANARGTFDPWDTQTWQPCTLEGIVSRNHDAFSVADFSHNVFKYVRKNHVKTTVHWKRHWQRARMAHEFVYGEQS"

nove_priznaky_dict = get_esm_embedding(sekvence_lig)

nova_data_df = pd.DataFrame([nove_priznaky_dict], columns=features_esm)
odhadnute_pH = best_rf_esm.predict(nova_data_df)

print(f"Predikované optimální pH pro ligasu je: {odhadnute_pH[0]:.2f}")

Predikované optimální pH pro ligasu je: 7.30


In [54]:
sekvence_ntp = "MIWQLTDDKRWSALRQRFSWVEEMHHTPQDPEHHGEGDVGVHTEMVLNALITLPEFQQLPAQQQEVLWAAVLLHDVEKRSTTVQENGRIQSPGHARRGELTARQILWRDIPTPFVLREQIVALVRLHGLPLWLLERPEPERLLLTAAMRIDTRLLALLARADLLGRQSPDQQSMLERIDLFELFCHEQQCWGKMRPFVSDSARWHYLTHEQSSPDFVPWGAEPFEVILLCGLPGMGKDRYISEQCQGIDVISLDDMRRRINASPDDKTATGRIVQQAKEEARVFLRQKKPFIWNATNITRQLRSQLISLFTAYGARVKIVYLEVPWAQWKQQNARREYAVPEAVVMRMASRLEVPQPDEAHSVEYRVIER"


nove_priznaky_dict = get_esm_embedding(sekvence_ntp)

nova_data_df = pd.DataFrame([nove_priznaky_dict], columns=features_esm)
odhadnute_pH = best_rf_esm.predict(nova_data_df)

print(f"Predikované optimální pH pro atpasu je: {odhadnute_pH[0]:.2f}")

Predikované optimální pH pro atpasu je: 7.53


In [40]:
sekvence_operon = "MNTQRKYGRTWHYPFSPGTTSDDRINTDYWQDLKAITQLVHTEKLDGENNCLNRYGVFARSHATPTQSAWTYKIRQRWQLLKNDLGDLELFGENLYAVHSIEYRALEQDFYLFAVRCQDMWLSWEEVQFYAALFDFPCVPEISGPQPGNDEKSWQRDFLALANARGTFDPWDTQTWQPCTLEGIVSRNHDAFSVADFSHNVFKYVRKNHVKTTVHWKRHWQRARMAHEFVYGEQSMIWQLTDDKRWSALRQRFSWVEEMHHTPQDPEHHGEGDVGVHTEMVLNALITLPEFQQLPAQQQEVLWAAVLLHDVEKRSTTVQENGRIQSPGHARRGELTARQILWRDIPTPFVLREQIVALVRLHGLPLWLLERPEPERLLLTAAMRIDTRLLALLARADLLGRQSPDQQSMLERIDLFELFCHEQQCWGKMRPFVSDSARWHYLTHEQSSPDFVPWGAEPFEVILLCGLPGMGKDRYISEQCQGIDVISLDDMRRRINASPDDKTATGRIVQQAKEEARVFLRQKKPFIWNATNITRQLRSQLISLFTAYGARVKIVYLEVPWAQWKQQNARREYAVPEAVVMRMASRLEVPQPDEAHSVEYRVIER"


nove_priznaky_dict = get_esm_embedding(sekvence_ntp)

nova_data_df = pd.DataFrame([nove_priznaky_dict], columns=features_esm)
odhadnute_pH = best_rf_esm.predict(nova_data_df)

print(f"Predikované optimální pH pro operon je: {odhadnute_pH[0]:.2f}")

Predikované optimální pH pro operon je: 7.57
